<a href="https://colab.research.google.com/github/zaku2590/classGCI/blob/main/comp2%E3%83%81%E3%83%A5%E3%83%BC%E3%83%8B%E3%83%B3%E3%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install optuna
!pip install catboost xgboost
!pip uninstall xgboost -y
!pip install xgboost --upgrade


Found existing installation: xgboost 3.0.2
Uninstalling xgboost-3.0.2:
  Successfully uninstalled xgboost-3.0.2
  Using cached xgboost-3.0.2-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
Using cached xgboost-3.0.2-py3-none-manylinux_2_28_x86_64.whl (253.9 MB)


In [7]:
# モジュールのインポート
import optuna
import lightgbm as lgb
import numpy as np  # 数値計算や配列操作を行うためのライブラリ
import pandas as pd  # 表形式のデータを扱うためのライブラリ
import matplotlib.pyplot as plt  # データ可視化のための基本的なグラフ描画ライブラリ
import seaborn as sns  # 高機能な統計グラフを描画するライブラリ
from sklearn.preprocessing import LabelEncoder  # カテゴリ変数を数値に変換するエンコーダ
from sklearn.ensemble import RandomForestClassifier  # ランダムフォレストによる分類器
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold ,cross_val_score # 層化K分割交差検証を行うクラス
from sklearn.metrics import roc_auc_score  # ROC AUCスコアを計算する評価指標
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [8]:
PATH = '/content/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')


In [9]:
# 使わない列の削除
train = train.drop(columns=["Id"])
test = test.drop(columns=["Id"])

cols_to_important = ['Age', 'Agility_3cone', 'Shuttle']

for data in cols_to_important:

    train[data + "_was_missing"] = train[data].isnull().astype(int)
    test[data + "_was_missing"] = test[data].isnull().astype(int)

# 平均で補完する対象の列
cols_to_fill = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']

# positionTypeで平均を埋める
for col in cols_to_fill:

    group_mean = train.groupby("Position_Type")[col].mean()
    train[col] = train[col].fillna(train["Position_Type"].map(group_mean))
    test[col] = test[col].fillna(test["Position_Type"].map(group_mean))

for col in ['Vertical_Jump', 'Sprint_40yd', 'Agility_3cone']:
    # 学習データ側で Position ごとの mean/std を事前に計算
    pos_stats = train.groupby("Position")[col].agg(["mean", "std"]).rename(columns={"mean": "mean_val", "std": "std_val"})

    # train にマージして計算
    train = train.merge(pos_stats, left_on="Position", right_index=True, how="left")
    train[f"{col}_z_by_pos"] = ((train[col] - train["mean_val"]) / train["std_val"]).fillna(0)
    train.drop(columns=["mean_val", "std_val"], inplace=True)

    # test も同様に train の mean/std を使用
    test = test.merge(pos_stats, left_on="Position", right_index=True, how="left")
    test[f"{col}_z_by_pos"] = ((test[col] - test["mean_val"]) / test["std_val"]).fillna(0)
    test.drop(columns=["mean_val", "std_val"], inplace=True)

train["BMI"] = train["Weight"] / (train["Height"] ** 2)
test["BMI"] = test["Weight"] / (test["Height"] ** 2)


train["Power_Index"] = train["Vertical_Jump"] * train["Weight"]
test["Power_Index"] = test["Vertical_Jump"] * test["Weight"]

train["Jump_per_kg"] = train["Vertical_Jump"] / train["Weight"]
test["Jump_per_kg"] = test["Vertical_Jump"] / test["Weight"]

train["Strength_per_kg"] = train["Bench_Press_Reps"] / train["Weight"]
test["Strength_per_kg"] = test["Bench_Press_Reps"] / test["Weight"]

# 総出力（パワー的な指標）
train["Total_Power"] = train["Bench_Press_Reps"] * train["Weight"]
test["Total_Power"] = test["Bench_Press_Reps"] * test["Weight"]

# 爆発力（ジャンプの距離 ÷ 走力）
train["Explosiveness_Index"] = train["Broad_Jump"] / train["Sprint_40yd"]
test["Explosiveness_Index"] = test["Broad_Jump"] / test["Sprint_40yd"]

train["Power_Ratio"] = train["Power_Index"] / train["BMI"]
test["Power_Ratio"] = test["Power_Index"] / test["BMI"]

# Agility_x_Strength
train["Agility_x_Strength"] = train["Agility_3cone"] * train["Strength_per_kg"]
test["Agility_x_Strength"] = test["Agility_3cone"] * test["Strength_per_kg"]

# Power_Index / Sprint_40yd: パワーをどれだけ速く出せるか
train["Power_to_Speed"] = train["Power_Index"] / train["Sprint_40yd"]
test["Power_to_Speed"] = test["Power_Index"] / test["Sprint_40yd"]

# BMI * Explosiveness_Index: 体格に対しての爆発力
train["BMI_x_Explosiveness"] = train["BMI"] * train["Explosiveness_Index"]
test["BMI_x_Explosiveness"] = test["BMI"] * test["Explosiveness_Index"]

# Strength_per_kg * Explosiveness_Index: 筋力と爆発力の掛け合わせ
train["Strength_x_Explosiveness"] = train["Strength_per_kg"] * train["Explosiveness_Index"]
test["Strength_x_Explosiveness"] = test["Strength_per_kg"] * test["Explosiveness_Index"]

# Sprint_40yd / Bench_Press_Reps: スピードと筋力のバランス（少ないほど優秀）
train["Speed_to_Strength_Ratio"] = train["Sprint_40yd"] / (train["Bench_Press_Reps"] + 1e-5)
test["Speed_to_Strength_Ratio"] = test["Sprint_40yd"] / (test["Bench_Press_Reps"] + 1e-5)

# Height * Agility_3cone: 身長と俊敏性の交差
train["Height_x_Agility"] = train["Height"] * train["Agility_3cone"]
test["Height_x_Agility"] = test["Height"] * test["Agility_3cone"]

# Strength_to_Speed = Bench_Press_Reps / Sprint_40yd
# train["Strength_to_Speed"] = train["Bench_Press_Reps"] / train["Sprint_40yd"]
# test["Strength_to_Speed"] = test["Bench_Press_Reps"] / test["Sprint_40yd"]

# # Height_to_Weight = Height / Weight
# train["Height_to_Weight"] = train["Height"] / train["Weight"]
# test["Height_to_Weight"] = test["Height"] / test["Weight"]

# Speed_x_Explosive
# train["Speed_x_Explosive"] = train["Sprint_40yd"] * train["Explosiveness_Index"]
# test["Speed_x_Explosive"] = test["Sprint_40yd"] * test["Explosiveness_Index"]

# BMI_x_Speed
# train["BMI_x_Speed"] = train["BMI"] * train["Sprint_40yd"]
# test["BMI_x_Speed"] = test["BMI"] * test["Sprint_40yd"]

# numeric_cols = train.select_dtypes(include=[np.number]).drop(columns=["Drafted"]).columns
# kmeans = KMeans(n_clusters=9, random_state=42, n_init=10)
# train["Cluster"] = kmeans.fit_predict(train[numeric_cols])
# test["Cluster"] = kmeans.predict(test[numeric_cols])

# # Cluster列をターゲットエンコーディング
# target_mean = train.groupby("Cluster")["Drafted"].mean()
# train["Cluster_TE"] = train["Cluster"].map(target_mean)
# test["Cluster_TE"] = test["Cluster"].map(target_mean)
# train = train.drop(columns=["Cluster"])
# test = test.drop(columns=["Cluster"])

all_data = pd.concat([train[["School"]], test[["School"]]])

# 各Schoolの出現回数をカウント
school_counts = all_data["School"].value_counts().to_dict()

# train にマップ
train["School_Count"] = train["School"].map(school_counts)

# test にマップ
test["School_Count"] = test["School"].map(school_counts)

train = train.drop(columns=["Shuttle", "Bench_Press_Reps", "School"])
test = test.drop(columns=["Shuttle", "Bench_Press_Reps", "School"])

train.head()

,Year,Age,Height,Weight,Sprint_40yd,Vertical_Jump,Broad_Jump,Agility_3cone,Player_Type,Position_Type,...,Total_Power,Explosiveness_Index,Power_Ratio,Agility_x_Strength,Power_to_Speed,BMI_x_Explosiveness,Strength_x_Explosiveness,Speed_to_Strength_Ratio,Height_x_Agility,School_Count
0,2011,21.0,1.9050,140.160042,5.39,59.69,251.46,7.910000,offense,offensive_lineman,...,4064.641227,46.653061,216.616502,1.636629,1552.161953,1801.832458,9.652814,0.185862,15.068550,2
1,2011,24.0,1.8288,87.089735,4.31,101.60,332.74,7.028157,offense,backs_receivers,...,1393.435761,77.201856,339.802159,1.291203,2052.973800,2010.306539,14.183413,0.269375,12.853094,3
2,2018,21.0,1.8542,92.986436,4.51,91.44,309.88,6.950000,offense,backs_receivers,...,929.864359,68.709534,314.375991,0.747421,1885.294832,1858.332634,7.389200,0.451000,12.886690,18
3,2010,21.0,1.9304,148.778297,5.09,76.20,254.00,8.120000,defense,defensive_lineman,...,5802.353599,49.901768,283.955045,2.128536,2227.290032,1992.328286,13.081000,0.130513,15.674848,15
4,2016,21.0,1.8796,92.079251,4.64,78.74,281.94,7.130000,offense,backs_receivers,...,1613.057418,60.762931,278.180244,1.356488,1562.569016,1583.687980,11.560190,0.264868,13.401548,39


In [10]:
import optuna
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# データ
X_base = train.drop(columns=["Drafted"])
y = train["Drafted"]
te_columns = ["Player_Type", "Position_Type", "Position"]

def optimize_model(model_name, n_trials=5):  # とにかく速く終わらせたいなら試行数を小さく
    def objective(trial):
        if model_name == "lgb":
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 31, 128),
                "max_depth": trial.suggest_int("max_depth", 3, 6),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
                "subsample": trial.suggest_float("subsample", 0.7, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
                "n_estimators": 300,
                "random_state": 2025,
                "verbosity": -1
            }
        elif model_name == "xgb":
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 6),
                "subsample": trial.suggest_float("subsample", 0.7, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
                "n_estimators": 300,
                "random_state": 2025,
                "use_label_encoder": False,
                "eval_metric": "auc",
                "verbosity": 0
            }
        elif model_name == "cat":
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth": trial.suggest_int("depth", 3, 6),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "iterations": 300,
                "eval_metric": "AUC",
                "random_state": 2025,
                "verbose": 0
            }
        else:
            raise ValueError("Invalid model_name")

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        aucs = []

        for train_idx, valid_idx in skf.split(X_base, y):
            X_train, X_valid = X_base.iloc[train_idx].copy(), X_base.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            for col in te_columns:
                te_map = y_train.groupby(X_train[col]).mean()
                X_train[f"{col}_TE"] = X_train[col].map(te_map).fillna(0)
                X_valid[f"{col}_TE"] = X_valid[col].map(te_map).fillna(0)
                X_train.drop(columns=[col], inplace=True)
                X_valid.drop(columns=[col], inplace=True)

            for df in [X_train, X_valid]:
                df.replace([np.inf, -np.inf], np.nan, inplace=True)
                df.fillna(0, inplace=True)

            if model_name == "lgb":
                model = LGBMClassifier(**params)
            elif model_name == "xgb":
                model = XGBClassifier(**params)
            elif model_name == "cat":
                model = CatBoostClassifier(**params)

            model.fit(X_train, y_train)
            preds = model.predict_proba(X_valid)[:, 1]
            aucs.append(roc_auc_score(y_valid, preds))

        mean_auc = np.mean(aucs)
        print(f"✅ [{model_name}] Mean AUC: {round(mean_auc, 4)}")
        return mean_auc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    print(f"⭐ {model_name.upper()} Best Params: {study.best_params}")
    print(f"⭐ {model_name.upper()} Best AUC: {round(study.best_value, 4)}")
    return study

# =============== 実行 ===============
study_lgb = optimize_model("lgb", n_trials=5)
study_xgb = optimize_model("xgb", n_trials=5)
study_cat = optimize_model("cat", n_trials=5)


[I 2025-07-04 08:25:52,939] A new study created in memory with name: no-name-05915869-f608-4eac-96ea-222c27432ed1
[I 2025-07-04 08:25:54,433] Trial 0 finished with value: 0.830240637380032 and parameters: {'learning_rate': 0.07760246747755509, 'num_leaves': 324, 'max_depth': 10, 'min_child_samples': 76, 'subsample': 0.5790588696805519, 'colsample_bytree': 0.8198289922318824, 'reg_alpha': 4.9160512856113686e-08, 'reg_lambda': 0.019930734232058586}. Best is trial 0 with value: 0.830240637380032.


✅ [lgb] Fold AUCs: [np.float64(0.7978), np.float64(0.86), np.float64(0.8537), np.float64(0.7911), np.float64(0.8487)] | Mean AUC: 0.8302


[I 2025-07-04 08:26:02,366] Trial 1 finished with value: 0.8286943561429118 and parameters: {'learning_rate': 0.003484444376510412, 'num_leaves': 116, 'max_depth': 7, 'min_child_samples': 62, 'subsample': 0.9014519657724036, 'colsample_bytree': 0.9611040175335215, 'reg_alpha': 0.0015262198948780303, 'reg_lambda': 0.0006879597427738934}. Best is trial 0 with value: 0.830240637380032.


✅ [lgb] Fold AUCs: [np.float64(0.7952), np.float64(0.8576), np.float64(0.8559), np.float64(0.7916), np.float64(0.843)] | Mean AUC: 0.8287


[I 2025-07-04 08:26:05,237] Trial 2 finished with value: 0.83199420133136 and parameters: {'learning_rate': 0.008950649860718588, 'num_leaves': 172, 'max_depth': 4, 'min_child_samples': 74, 'subsample': 0.980051273625464, 'colsample_bytree': 0.5970178259404151, 'reg_alpha': 0.9268954604084223, 'reg_lambda': 0.0005094798476503878}. Best is trial 2 with value: 0.83199420133136.


...

[I 2025-07-04 08:27:00,404] Trial 28 finished with value: 0.8322057209487956 and parameters: {'learning_rate': 0.02704549190880387, 'num_leaves': 796, 'max_depth': 3, 'min_child_samples': 41, 'subsample': 0.6144229099339369, 'colsample_bytree': 0.7133004438789297, 'reg_alpha': 3.5979490195031646e-06, 'reg_lambda': 0.003799040047028343}. Best is trial 20 with value: 0.835631505745277.


✅ [lgb] Fold AUCs: [np.float64(0.8023), np.float64(0.8543), np.float64(0.8665), np.float64(0.7945), np.float64(0.8433)] | Mean AUC: 0.8322


[I 2025-07-04 08:27:02,364] Trial 29 finished with value: 0.8310113497064426 and parameters: {'learning_rate': 0.014680892856652701, 'num_leaves': 367, 'max_depth': 5, 'min_child_samples': 77, 'subsample': 0.8299468702061357, 'colsample_bytree': 0.5714275556112314, 'reg_alpha': 1.5505640333567204e-08, 'reg_lambda': 0.03791174475418238}. Best is trial 20 with value: 0.835631505745277.
[I 2025-07-04 08:27:02,366] A new study created in memory with name: no-name-5e899159-37ca-4d60-8079-db741dc32d7c
[W 2025-07-04 08:27:02,387] Trial 0 failed with parameters: {'learning_rate': 0.05572885343440992, 'max_depth': 3, 'subsample': 0.7006948268297926, 'colsample_bytree': 0.9497655290342452, 'reg_alpha': 0.00010292488231804026, 'reg_lambda': 0.006796154573901851} because of the following error: TypeError("XGBClassifier.fit() got an unexpected keyword argument 'callbacks'").
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 201, in _

✅ [lgb] Fold AUCs: [np.float64(0.7977), np.float64(0.8627), np.float64(0.8534), np.float64(0.7973), np.float64(0.8439)] | Mean AUC: 0.831

⭐ LGB Best Params: {'learning_rate': 0.045916098781752526, 'num_leaves': 430, 'max_depth': 3, 'min_child_samples': 87, 'subsample': 0.7373722028759282, 'colsample_bytree': 0.5772312732058846, 'reg_alpha': 2.0509338240375774, 'reg_lambda': 0.9123578614875963}
⭐ LGB Best AUC: 0.8356



TypeError: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'